# Prompt Engineering — NormaBot RAG

## Introducción

NormaBot utiliza un pipeline **Corrective RAG** con tres pasos:

1. **Retrieve** — recupera documentos de ChromaDB
2. **Grade** — filtra cuáles son relevantes para la pregunta (Ollama Qwen 2.5 3B)
3. **Generate** — genera la respuesta con citas legales (Bedrock Nova Lite)

Cada paso tiene un **prompt** que controla el comportamiento del LLM. Este notebook documenta el proceso de iteración sobre los prompts:

- Medimos el baseline (prompts actuales)
- Creamos 2-3 variantes con técnicas distintas
- Comparamos con métricas fijas
- Decidimos qué variante se queda y por qué

### Prompts analizados

| Prompt | LLM | Métrica principal |
|--------|-----|------------------|
| `GRADING_PROMPT` | Qwen 2.5 3B (Ollama) | Precision / Recall |
| `GENERATE_PROMPT` | Nova Lite (Bedrock) | Faithfulness / Answer Relevancy (RAGAS) |

## 1. Prompt de Grading

### ¿Qué hace?

Recibe un documento recuperado de ChromaDB y la pregunta del usuario.
Decide si el documento contiene información útil para responder — devuelve `si` o `no`.

### ¿Por qué importa?

Un grader demasiado **estricto** (recall bajo) descarta documentos útiles → el generador tiene menos contexto.
Un grader demasiado **permisivo** (precision baja) pasa basura al generador → riesgo de alucinaciones.

En un sistema legal como NormaBot, **el recall es prioritario**: preferimos pasar un documento dudoso antes de perder una cita legal relevante.

### Prompt v0 — Zero-shot (actual en producción)

```
Dado el siguiente documento y la pregunta,
¿el documento contiene información útil para responder la pregunta?

Documento: {document}
Pregunta: {query}

Responde solo con "si" o "no":
```

**Técnica**: Zero-shot — sin ejemplos, instrucción directa.

In [1]:
import json
from pathlib import Path

RESULTS_PATH = Path("../../eval/grading_results.json")

with open(RESULTS_PATH, encoding="utf-8") as f:
    results = json.load(f)

print("Variantes disponibles:", list(results.keys()))

Variantes disponibles: ['v0', 'v1', 'v2']


In [2]:
# Métricas baseline (v0)
v0 = results["v0"]["metrics"]
print("=== Baseline — Prompt v0 (Zero-shot) ===")
print(f"  Precision : {v0['precision']:.4f}")
print(f"  Recall    : {v0['recall']:.4f}")
print(f"  Accuracy  : {v0['accuracy']:.4f}")
print(f"  TP={v0['tp']}  FP={v0['fp']}  FN={v0['fn']}  Total={v0['n_total']}")

=== Baseline — Prompt v0 (Zero-shot) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


### Análisis del baseline

| Métrica | Valor | Interpretación |
|---------|-------|----------------|
| Precision | 1.000 | Nunca acepta documentos irrelevantes (0 falsos positivos) |
| Recall | 0.545 | Descarta casi la mitad de los documentos relevantes (5 falsos negativos) |
| Accuracy | 0.792 | Acierta en 19 de 24 casos |

**Problema identificado**: El prompt v0 es demasiado conservador. Descarta documentos útiles, reduciendo el contexto disponible para el generador.

## 2. Variante v1 — Few-shot

**Técnica**: Añadir 2 ejemplos al prompt (1 relevante + 1 irrelevante) para guiar al modelo.

**Hipótesis**: Al ver ejemplos concretos, el modelo entenderá mejor qué significa "útil para responder".

```
Tu tarea es decidir si un documento contiene información útil para responder una pregunta.

Ejemplos:

Pregunta: ¿Qué prácticas de IA están prohibidas?
Documento: Artículo 5. Quedan prohibidas las siguientes prácticas de IA...
Respuesta: si

Pregunta: ¿Qué prácticas de IA están prohibidas?
Documento: Artículo 9. Se establecerá un sistema de gestión de riesgos...
Respuesta: no

Ahora evalúa:

Pregunta: {query}
Documento: {document}

Responde solo con "si" o "no":
```

In [3]:
v1 = results["v1"]["metrics"]
print("=== Variante v1 (Few-shot) ===")
print(f"  Precision : {v1['precision']:.4f}")
print(f"  Recall    : {v1['recall']:.4f}")
print(f"  Accuracy  : {v1['accuracy']:.4f}")
print(f"  TP={v1['tp']}  FP={v1['fp']}  FN={v1['fn']}  Total={v1['n_total']}")

=== Variante v1 (Few-shot) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


## 3. Variante v2 — Chain of Thought

**Técnica**: Pedir al modelo que razone antes de decidir.

**Hipótesis**: Al forzar al modelo a identificar primero el tema del documento, mejora su capacidad de relacionarlo con la pregunta.

```
Dado el siguiente documento y la pregunta, determina si el documento contiene
información útil para responder la pregunta.

Documento: {document}
Pregunta: {query}

Primero identifica de qué trata el documento en una frase.
Luego decide si esa información ayuda a responder la pregunta.
Termina tu respuesta con "Veredicto: si" o "Veredicto: no".
```

**Nota técnica**: Esta variante requiere `num_predict=150` en lugar de 10, ya que el modelo necesita generar texto de razonamiento antes del veredicto.

In [4]:
v2 = results["v2"]["metrics"]
print("=== Variante v2 (Chain of Thought) ===")
print(f"  Precision : {v2['precision']:.4f}")
print(f"  Recall    : {v2['recall']:.4f}")
print(f"  Accuracy  : {v2['accuracy']:.4f}")
print(f"  TP={v2['tp']}  FP={v2['fp']}  FN={v2['fn']}  Total={v2['n_total']}")

=== Variante v2 (Chain of Thought) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


## 4. Tabla comparativa

In [5]:
print(f"{'Métrica':<15} {'v0 (zero-shot)':>15} {'v1 (few-shot)':>15} {'v2 (CoT)':>15}")
print("-" * 62)
for metric in ["precision", "recall", "accuracy"]:
    print(f"{metric:<15} {results['v0']['metrics'][metric]:>15.4f} {results['v1']['metrics'][metric]:>15.4f} {results['v2']['metrics'][metric]:>15.4f}")

print()
print("Casos que fallan por variante:")
for variant in ["v0", "v1", "v2"]:
    fallos = [i+1 for i, r in enumerate(results[variant]["detail"]) if not r["correct"]]
    print(f"  {variant}: pares {fallos}")

Métrica          v0 (zero-shot)   v1 (few-shot)        v2 (CoT)
--------------------------------------------------------------
precision                1.0000          1.0000          1.0000
recall                   0.5455          0.5455          0.5455
accuracy                 0.7917          0.7917          0.7917

Casos que fallan por variante:
  v0: pares [1, 7, 11, 17, 21]
  v1: pares [7, 11, 15, 17, 21]
  v2: pares [7, 9, 11, 13, 21]


## 5. Análisis de errores

Los tres prompts obtienen el mismo recall (0.545) pero **fallan en casos distintos**:

| Par | Pregunta | Documento | v0 | v1 | v2 |
|-----|----------|-----------|----|----|----|
| 7  | ¿Qué multas prevé el EU AI Act? | Art. 71 (infracciones) | ❌ | ❌ | ❌ |
| 11 | ¿Requisitos de gobernanza de datos? | Art. 10 (conjuntos de datos) | ❌ | ❌ | ❌ |
| 21 | ¿Transparencia para riesgo limitado? | Art. 52 (informar a usuarios) | ❌ | ❌ | ❌ |

### Patrón de fallo

Los casos que fallan consistentemente comparten una característica: el documento usa **vocabulario indirecto** respecto a la pregunta:

- "multas" (pregunta) vs "infracciones" (documento)
- "gobernanza de datos" (pregunta) vs "conjuntos de datos de entrenamiento" (documento)  
- "riesgo limitado" (pregunta) vs el Art. 52 que no menciona explícitamente esa categoría

### Conclusión del análisis

El problema no está en el prompt sino en la **capacidad del modelo**: Qwen 2.5 3B no tiene suficiente comprensión semántica para relacionar términos sinónimos en contexto legal. Cambiar el prompt no resuelve esta limitación.

## 6. Decisión final

### Qué variante se queda

**Se mantiene v0 (Zero-shot)** por las siguientes razones:

1. **Ninguna variante mejoró el recall**: las tres obtienen recall=0.545 con 5 falsos negativos
2. **v1 intercambió errores**: arregló el par 1 pero rompió el par 15 — resultado neto igual
3. **v2 empeoró en casos fáciles**: con Chain of Thought, el modelo falló en pares (9, 13) que v0 resolvía correctamente
4. **Coste computacional**: v2 genera ~15x más tokens por llamada sin beneficio en métricas

### Recomendación para el equipo

El recall bajo (0.545) es una **limitación del modelo 3B**, no del prompt. Las opciones para mejorarlo son:

- Aumentar `k` en el retriever (recuperar más documentos) para compensar los falsos negativos del grader
- Usar un modelo más grande para grading (mayor coste de inferencia)
- Ajustar el umbral del fallback por score (`_grade_by_score`) como complemento al grading LLM

Estas decisiones afectan `src/rag/main.py` y deben consensuarse con el equipo.

---

## 7. Prompt de Generate

### ¿Qué hace?

Recibe los documentos relevantes (ya filtrados por el grader) y la pregunta del usuario.
Genera una respuesta con citas legales exactas usando Bedrock Nova Lite.

### ¿Por qué importa?

En un sistema legal, la respuesta debe estar **fundamentada** en los documentos (faithfulness alta) y ser **relevante** para la pregunta (answer_relevancy alta). Las alucinaciones son inaceptables.

### Métricas usadas (RAGAS)

| Métrica | Umbral | ¿Qué mide? |
|---------|--------|-----------|
| `faithfulness` | ≥ 0.80 | ¿La respuesta está fundamentada en los documentos? |
| `answer_relevancy` | ≥ 0.85 | ¿La respuesta es relevante a la pregunta? |

### Prompt v0 — Role + grounding básico (actual en producción)

```
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde la pregunta usando SOLO la informacion de los documentos proporcionados.
Cita siempre la ley y articulo exactos. Si no hay informacion suficiente, dilo.
No inventes articulos ni citas que no aparezcan en los documentos.

Documentos:
{context}

Pregunta: {query}

Responde de forma clara y estructurada. Cita las fuentes al final.
```

**Técnica**: Role prompting + instrucción de grounding explícita.

## 13. Resumen global

### Decisiones tomadas

| Componente | Prompt elegido | Técnica | Métrica |
|-----------|---------------|---------|---------|
| **Grading** | v0 (Zero-shot) | Role prompting | Precision=1.0 / Recall=0.545 |
| **Generate** | v1 (Estructurado) | Role + grounding + formato | Qualitativa + RAGAS en CI/CD |
| **Orquestador** | Actual | ReAct system prompt | 8/8 routing correcto (cualitativo) |

### Limitación conocida

El recall del grading (0.545) es la principal limitación del pipeline. No se debe al prompt sino a la **capacidad del modelo**: Qwen 2.5 3B no relaciona términos sinónimos en contexto legal con suficiente fiabilidad. Esta limitación está documentada y el equipo puede compensarla aumentando `k` en el retriever.

### Próximo paso

La evaluación RAGAS del prompt de generate (`eval/run_ragas.py`) completará la validación cuantitativa del pipeline cuando se ejecute en CI/CD con acceso a Bedrock.

In [ ]:
# Evaluación cualitativa del orquestador
# Resultado de 8 queries de prueba manuales con Bedrock Nova Lite

evaluacion_orquestador = [
    {
        "query": "¿Qué prácticas de IA están prohibidas según el EU AI Act?",
        "tipo_esperado": "normativa",
        "herramienta_esperada": "rag_normativo",
        "resultado": "RAG invocado correctamente — cita Art. 5",
        "routing_correcto": True,
    },
    {
        "query": "¿Cuándo un sistema de IA es de alto riesgo?",
        "tipo_esperado": "normativa",
        "herramienta_esperada": "rag_normativo",
        "resultado": "RAG invocado correctamente — cita Art. 6 y Anexo III",
        "routing_correcto": True,
    },
    {
        "query": "Clasifica este sistema: sistema de reconocimiento facial en aeropuertos",
        "tipo_esperado": "clasificacion",
        "herramienta_esperada": "clasificador_riesgo",
        "resultado": "Clasificador invocado — respuesta pendiente de integración real",
        "routing_correcto": True,
    },
    {
        "query": "Genera un informe de cumplimiento para un chatbot de atención al cliente",
        "tipo_esperado": "informe",
        "herramienta_esperada": "generar_informe",
        "resultado": "Informe invocado — respuesta pendiente de integración real",
        "routing_correcto": True,
    },
    {
        "query": "¿Qué multas prevé el EU AI Act?",
        "tipo_esperado": "normativa",
        "herramienta_esperada": "rag_normativo",
        "resultado": "RAG invocado correctamente — cita Art. 71",
        "routing_correcto": True,
    },
    {
        "query": "Explícame el sistema de gestión de riesgos",
        "tipo_esperado": "normativa",
        "herramienta_esperada": "rag_normativo",
        "resultado": "RAG invocado correctamente — cita Art. 9",
        "routing_correcto": True,
    },
    {
        "query": "¿Qué obligaciones tiene un proveedor de IA de alto riesgo?",
        "tipo_esperado": "normativa",
        "herramienta_esperada": "rag_normativo",
        "resultado": "RAG invocado correctamente — cita Arts. 9-16",
        "routing_correcto": True,
    },
    {
        "query": "Hola, ¿en qué me puedes ayudar?",
        "tipo_esperado": "saludo/ambigua",
        "herramienta_esperada": "ninguna",
        "resultado": "Responde directamente sin invocar herramienta — describe capacidades",
        "routing_correcto": True,
    },
]

correctos = sum(1 for r in evaluacion_orquestador if r["routing_correcto"])
total = len(evaluacion_orquestador)

print(f"Evaluación cualitativa del orquestador: {correctos}/{total} routing correcto")
print()
print(f"{'Query':<55} {'Esperado':<22} {'Correcto'}")
print("─" * 85)
for r in evaluacion_orquestador:
    icono = "✓" if r["routing_correcto"] else "✗"
    print(f"{r['query'][:54]:<55} {r['herramienta_esperada']:<22} {icono}")

---

## 12. Prompt del Orquestador

### ¿Qué hace?

El orquestador (`src/orchestrator/main.py`) es un agente ReAct (LangGraph) que:
1. Recibe la pregunta del usuario
2. Decide qué herramienta invocar: RAG normativo, clasificador de riesgo o informes
3. Llama a la herramienta y genera una respuesta final

El prompt del orquestador es el system prompt que configura este comportamiento de decisión.

### Métrica usada

Para el orquestador se usa evaluación **cualitativa** (no cuantitativa), ya que su función principal es de routing/decisión, no de generación:

| Criterio | Descripción |
|----------|-------------|
| **Routing correcto** | ¿Invoca la herramienta adecuada para cada tipo de pregunta? |
| **Preguntas normativas** | → debe invocar RAG |
| **Preguntas de clasificación** | → debe invocar clasificador |
| **Preguntas de informe** | → debe invocar informes |
| **Preguntas ambiguas** | → debe manejar con gracia o pedir aclaración |

## 11. Decisión final — Prompt de Generate

### Qué variante se recomienda

**Se recomienda v1 (Formato estructurado)** como la variante a evaluar con RAGAS en CI/CD:

1. **Formato consistente**: La separación explícita en `## Respuesta` / `## Fuentes` hace las citas más fáciles de verificar por el usuario y de parsear programáticamente.
2. **Coste razonable**: Solo ~25 tokens adicionales respecto a v0 — coste marginal con Nova Lite.
3. **Sin riesgos del few-shot**: v2 añade ~130 tokens y puede sesgar respuestas hacia el ejemplo incluido. Para un sistema legal donde la precisión importa, el riesgo no justifica el beneficio potencial.
4. **Grounding igual de fuerte**: Ambas variantes (v0, v1) mantienen las instrucciones anti-hallucination sin cambios.

### Decisión final del pipeline

| Paso | Prompt elegido | Razón |
|------|---------------|-------|
| **Grading** | v0 (Zero-shot) | Ninguna variante mejoró recall; se mantiene el más simple |
| **Generate** | v1 (Estructurado) | Mismo grounding, mejor legibilidad de fuentes |

### Pendiente (CI/CD)

La validación cuantitativa con RAGAS de `eval/run_ragas.py` confirmará o revisará esta decisión:
- **faithfulness ≥ 0.80**: ¿La respuesta está fundamentada en los documentos?
- **answer_relevancy ≥ 0.85**: ¿La respuesta es relevante a la pregunta?

## 10. Tabla comparativa — Prompt de Generate

> **Nota**: La evaluación cuantitativa con RAGAS (faithfulness, answer_relevancy) se ejecuta en el pipeline de CI/CD (`eval/run_ragas.py`) donde Bedrock está disponible. Los umbrales objetivo son faithfulness ≥ 0.80 y answer_relevancy ≥ 0.85.

La siguiente tabla resume las diferencias entre variantes y la evaluación **cualitativa** basada en respuestas reales de Bedrock Nova Lite:

| Criterio | v0 (Zero-shot) | v1 (Estructurado) | v2 (Few-shot) |
|----------|---------------|-------------------|---------------|
| **Técnica** | Role + grounding | Role + grounding + formato | Role + grounding + ejemplo |
| **Tokens aprox.** | ~120 | ~145 | ~250 |
| **Citas legales** | Correctas pero inline | Correctas, sección separada | Correctas, sección separada |
| **Formato de salida** | Variable | Consistente (## Respuesta / ## Fuentes) | Consistente |
| **Parseable programáticamente** | No | Sí | Sí |
| **Riesgo de hallucinations** | Bajo | Bajo | Bajo |
| **Alineación con grounding** | Alta | Alta | Alta |
| **Evaluación RAGAS (CI/CD)** | Pendiente | Pendiente | Pendiente |

In [ ]:
# Prompt v2 — Few-shot con ejemplo de respuesta ideal
GENERATE_PROMPT_V2 = """\
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde SOLO con informacion de los documentos proporcionados. Cita ley y articulo exactos.
Si no hay informacion suficiente, dilo. No inventes citas.

Ejemplo de respuesta ideal:

Pregunta: ¿Qué sistemas de IA son de alto riesgo?
Contexto: [1] EU AI Act — Artículo 6
Los sistemas de IA comprendidos en el Anexo III y que presenten riesgo significativo
para la salud, la seguridad o los derechos fundamentales se consideran de alto riesgo.

Respuesta:
## Respuesta
Los sistemas de IA de alto riesgo son aquellos que cumplen dos condiciones (Art. 6 EU AI Act):
1. Están incluidos en el Anexo III del Reglamento.
2. Presentan riesgo significativo para salud, seguridad o derechos fundamentales.

## Fuentes
- EU AI Act, Art. 6 — Clasificación de sistemas de alto riesgo

---
Ahora responde:

Documentos:
{context}

Pregunta: {query}"""

print("Prompt v2 — Few-shot con ejemplo ideal")
print("─" * 50)
print("Técnica: Role prompting + grounding + ejemplo de respuesta completa")
print("Tokens aprox.: 250 (sin contexto)")
print()
print("Ventajas sobre v1:")
print("  ✓ El modelo ve un ejemplo de calidad → ancla el nivel de detalle esperado")
print("  ✓ El formato de citas queda demostrado, no solo descrito")
print()
print("Desventajas:")
print("  ~ ~100 tokens adicionales por llamada")
print("  ~ El ejemplo puede sesgar la respuesta si no es representativo")

## 9. Variante v2 — Few-shot con ejemplo ideal

**Técnica**: Incluir un ejemplo completo de pregunta + contexto + respuesta ideal en el prompt.

**Hipótesis**: Ver una respuesta de alta calidad como ejemplo ancla el estilo, el nivel de detalle, y el formato de las citas que se espera del modelo.

```
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde SOLO con información de los documentos. Cita ley y artículo exactos.

Ejemplo de respuesta ideal:

Pregunta: ¿Qué sistemas de IA son de alto riesgo?
Contexto: [1] EU AI Act — Art. 6\nLos sistemas de IA comprendidos en el Anexo III...
Respuesta:
## Respuesta
Son de alto riesgo los sistemas del Anexo III que presenten riesgo para salud,
seguridad o derechos fundamentales (Art. 6 EU AI Act).
## Fuentes
- EU AI Act, Art. 6 — Clasificación de alto riesgo

---
Ahora responde:

Documentos:
{context}

Pregunta: {query}
```

**Cambio respecto a v1**: Añade un ejemplo completo de respuesta ideal para guiar el estilo.

In [ ]:
# Prompt v1 — Formato estructurado
GENERATE_PROMPT_V1 = """\
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde la pregunta usando SOLO la informacion de los documentos proporcionados.
Cita siempre la ley y articulo exactos. Si no hay informacion suficiente, dilo.
No inventes articulos ni citas que no aparezcan en los documentos.

Documentos:
{context}

Pregunta: {query}

Responde siguiendo EXACTAMENTE este formato:

## Respuesta
[Tu respuesta aquí, clara y concisa]

## Fuentes
- [Ley] Art. [N] — [Descripción breve]"""

# Respuesta de ejemplo con el prompt v1
RESPUESTA_V1 = """
## Respuesta
El EU AI Act prohíbe explícitamente varias prácticas de IA en su Artículo 5:

- Sistemas que utilicen técnicas subliminales para alterar el comportamiento humano
  causando perjuicio físico o psicológico.
- Sistemas que exploten vulnerabilidades de grupos específicos (edad, discapacidad,
  situación socioeconómica) para distorsionar su comportamiento.

## Fuentes
- EU AI Act, Art. 5 — Prácticas de IA prohibidas

_Informe preliminar generado por IA. Consulte profesional jurídico._
"""

print("Prompt v1 — Formato estructurado")
print("─" * 50)
print("Técnica: Role prompting + grounding + formato de salida explícito")
print("Tokens aprox.: 145 (sin contexto)")
print()
print("Respuesta de ejemplo:")
print(RESPUESTA_V1)

print("Ventajas sobre v0:")
print("  ✓ Fuentes separadas en sección propia → más fáciles de verificar")
print("  ✓ Estructura consistente para parsear programáticamente")
print("  ~ Más tokens en el prompt (coste marginal bajo con Nova Lite)")

## 8. Variante v1 — Formato estructurado

**Técnica**: Especificar el formato de salida esperado con secciones explícitas.

**Hipótesis**: Al indicar la estructura de la respuesta (## Respuesta + ## Fuentes), el modelo separa mejor el contenido de las citas, haciendo las fuentes más legibles y verificables.

```
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde la pregunta usando SOLO la informacion de los documentos proporcionados.
Cita siempre la ley y articulo exactos. Si no hay informacion suficiente, dilo.
No inventes articulos ni citas que no aparezcan en los documentos.

Documentos:
{context}

Pregunta: {query}

Responde siguiendo EXACTAMENTE este formato:

## Respuesta
[Tu respuesta aquí, clara y concisa]

## Fuentes
- [Ley] Art. [N] — [Descripción breve]
```

**Cambio respecto a v0**: Añade instrucción de formato estructurado con secciones separadas.

In [ ]:
# Prompt v0 — el actual en src/rag/main.py
GENERATE_PROMPT_V0 = """\
Eres un asistente juridico especializado en el EU AI Act y normativa española de IA.
Responde la pregunta usando SOLO la informacion de los documentos proporcionados.
Cita siempre la ley y articulo exactos. Si no hay informacion suficiente, dilo.
No inventes articulos ni citas que no aparezcan en los documentos.

Documentos:
{context}

Pregunta: {query}

Responde de forma clara y estructurada. Cita las fuentes al final."""

# Respuesta real de Bedrock Nova Lite con el prompt v0 (verificada manualmente)
RESPUESTA_V0 = """
Según el EU AI Act, las prácticas de IA prohibidas se recogen en el Artículo 5:

a) Sistemas que utilicen técnicas subliminales para alterar el comportamiento de una
   persona de manera que cause o pueda causar perjuicio físico o psicológico.
b) Sistemas que exploten vulnerabilidades de grupos específicos de personas debido
   a su edad, discapacidad o situación socioeconómica.

Fuentes:
[1] EU AI Act — Artículo 5 (Prácticas prohibidas)

_Informe preliminar generado por IA. Consulte profesional jurídico._
"""

print("Prompt v0 — Role + grounding")
print("─" * 50)
print("Técnica: Role prompting + instrucción explícita de grounding")
print("Tokens aprox.: 120 (sin contexto)")
print()
print("Respuesta de ejemplo (Bedrock Nova Lite):")
print(RESPUESTA_V0)